# To dos:
- create subagents + permissions
- add backend
- add skill + middleware 
- tools: git commit
- SANDBOX + HITL

In [ ]:
from deepagents import create_deep_agent,FilesystemPermission
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.store.memory import InMemoryStore

# Create subagent
# ingest subagent, skills = paper ingestion, wiki maintenance, citation extract 
ingest_subagent = {
    "name": "ingest",
    "permissions": [
        FilesystemPermission(paths=["/raw/**"],      operations=["read"],         mode="allow"),
        FilesystemPermission(paths=["/wiki/**"],     operations=["read","write"],  mode="allow"),
        FilesystemPermission(paths=["/workspace/**"],operations=["read","write"],  mode="allow"),
        FilesystemPermission(paths=["/**"],          operations=["read","write"],  mode="deny"),
    ],
    "description": "Paper ingestion, wiki maintenance, citation extract",
    "system_prompt": "You are a paper ingestion agent",
    "tools": [],
    #"model": "openai:gpt-5.2",  # Optional override, defaults to main agent model
}



# query subagent
# code subagent

agent = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    skills=["/skills/"],
    memory=["/memories/AGENTS.md"],   # wiki schema
    system_prompt="""You are Paper2Wiki, a research knowledge
    base agent. You build and maintain a structured wiki from
    research papers following the Karpathy LLM Wiki pattern.
    Always follow the schema in your AGENTS.md memory file.""",
    backend=CompositeBackend(
        default=StateBackend(),
        routes={
            # Agent-scoped: same for all users
            "/memories/AGENTS.md": StoreBackend(
                namespace=lambda rt: (rt.server_info.assistant_id,)
            ),
            "/skills/": StoreBackend(
                namespace=lambda rt: (rt.server_info.assistant_id,)
            ),

            # User-scoped: isolated per user
            "/memories/preferences.md": StoreBackend(
                namespace=lambda rt: (rt.server_info.user.identity,)
            ),
            "/wiki/": StoreBackend(
                namespace=lambda rt: (rt.server_info.user.identity,)
            ),
            "/raw/": StoreBackend(
                namespace=lambda rt: (rt.server_info.user.identity,)
            ),
        }
    ),
    store=InMemoryStore(),
    subagents=subagents
        
)



In [ ]:
# ingest
@tool
def fetch_arxiv(query: str) -> dict:
    """Search and fetch paper by arXiv ID, URL, or topic name.
    Returns: {title, authors, abstract, pdf_path, metadata}
    e.g. "attention is all you need" → finds + downloads paper""" 

@tool  
def parse_pdf(path: str) -> dict:
    """Extract structured content from PDF.
    Returns: {title, abstract, sections, figures, tables, references}"""

@tool
def fetch_web_article(url: str) -> dict:
    """Fetch web article and convert to markdown (Obsidian Web Clipper equivalent).
    Returns: {title, content_md, images: [urls]}"""

# git
@tool
def git_commit_and_push(message: str) -> str:
    """Commit and push wiki changes. Requires HITL approval."""
    # interrupt() called inside
    ...

# Image handling
@tool
def download_image(url: str, save_path: str) -> str:
    """Download image from URL to raw/assets/."""
    ...

# Lint
@tool
def run_lint(wiki_dir: str = "/wiki") -> dict:
    """Run wiki health check. Returns {passed: bool, issues: list}."""
    # runs lint.py programmatically
    ...

# Marp (Code subagent)
@tool
def render_marp(input_path: str, output_path: str) -> str:
    """Render marp markdown to PDF/HTML slides via sandbox."""
    # execute("marp {input} --output {output}")
    ...

# Frontmatter validation
@tool  
def validate_frontmatter(path: str) -> dict:
    """Validate YAML frontmatter against wiki schema."""
    # checks required fields, types, values
    ...

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "What is langgraph?"}]})

# Print the agent's response
print(result["messages"][-1].content)